# 双色球量化研究（Kaggle 一键运行，未发版功能）

取数 → 概率与显著性分析 → 多算法对照回测 → （可选）LSTM → 单文件 HTML 仪表盘 → （可选）Gradio 界面。

**本 Notebook 不承诺预测。** 每个算法都与随机基线在同一窗口、同一注数、同一成本下逐期对比；
每个「高频号码」排名都附 95% 置信区间与卡方结论。一等奖概率恒为 `1 / 17,721,088`，与选号方式无关。

## 开关一览

| 开关 | 默认 | 作用 | 代价 |
|---|---|---|---|
| `SEED` | `None` | 运行时随机生成并打印；要复现把打印值填回 | 无 |
| `WINDOW` | `100` | 回测的信息窗口（期） | 无 |
| `MAX_PERIODS` | `600` | 回测期数上限，`0` 为全部 | 越大越慢 |
| `RUN_ARIMA` | `False` | 追加 ARIMA 评分器 | **很慢**，600 期可能几十分钟 |
| `RUN_LSTM` | `False` | 追加 LSTM 顺序回测与最终候选 | Kaggle CPU 上数分钟 |
| `RUN_WEBAPP` | `False` | 启动 Gradio，`share=True` 对外 | 需 gradio |
| `SETUP_CJK_FONT` | `True` | 现场安装中文字体，让图表显示中文 | 约 20~40 秒 |
| `DATA_CSV` | `None` | 自动取数失败时的本地 CSV 兜底 | 无 |

## 预计耗时（默认配置）

取数约 5 秒，概率分析约 1 秒，多算法回测（9 个策略 × 600 期）约 1~3 分钟，报告生成约 10 秒。
即默认全流程 **3 分钟以内**，不含 ARIMA 与 LSTM。


In [ ]:
# ===== 用户配置 =====
SEED = None            # None = 每次运行随机生成并打印；要复现就把打印出的 SEED 填回来
WINDOW = 100           # 回测的信息窗口（期）
MAX_PERIODS = 600      # 回测期数上限，0 表示全部（全量较慢）
RUN_ARIMA = False      # 闭式 CSS 估计，不需要 statsmodels，只多花 10~30 秒
SETUP_CJK_FONT = True  # Kaggle 默认无中文字体；True 会现场 apt 安装（约 20~40 秒）
RUN_LSTM = False       # 需要 tensorflow；追加顺序回测与最终候选
LSTM_EPOCHS = 10       # 仅在 RUN_LSTM=True 时生效
LSTM_MAX_PREDICTIONS = 300
RUN_WEBAPP = False     # 需要 gradio；Kaggle 不暴露端口，必须 share=True
DATA_CSV = None        # 自动取数失败时指向本地 CSV（Kaggle Dataset 路径）

REPO_URL = 'https://github.com/wangmou23/ssq-quant-research-system.git'
REPO_DIR = '/kaggle/working/ssq-quant-research-system'

import time
if SEED is None:
    SEED = int(time.time_ns() % (2**31))
print(f'实际使用 SEED={SEED}（本次运行以此为准；要复现请把该值填回 SEED）')
print('配置完成。SEED 同时控制基线回测的取号、LSTM 权重初始化与蓝球抽样。')


In [ ]:
# ===== 加载源码 =====
import os, sys, subprocess, pathlib, shutil
repo_dir = pathlib.Path(REPO_DIR)
if repo_dir.exists() and not (repo_dir / 'ssq_research').exists():
    shutil.rmtree(repo_dir)
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
else:
    # 目录已存在（重跑，或另一份 Notebook 克隆到同一路径）时必须强制对齐远端，
    # 否则会静默复用旧代码：跑出来的结果和仓库里看到的不是同一份。
    fetched = subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth', '1', 'origin', 'main'],
                             capture_output=True, text=True)
    if fetched.returncode == 0:
        subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'FETCH_HEAD'], check=True)
        print('已把已有目录更新到远端 main')
    else:
        print('警告：已有目录无法更新，继续用现有代码 ->', fetched.stderr.strip()[:200])
sys.path.insert(0, str(repo_dir))
print('源码目录:', repo_dir)
print('提交:', subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], text=True).strip())
import ssq_research
print('版本:', ssq_research.__version__)


In [ ]:
# ===== 依赖检查 =====
# 不执行 pip install -e：用 sys.path 引入本仓库即可，装包只会污染 Kaggle 环境。
# 注意：available_algorithms() 会真的 import tensorflow / statsmodels 去探测，
# 首次约 10~30 秒，属正常现象，不是卡住。
import platform, importlib.util
print('Python:', platform.python_version())
for name in ('numpy', 'matplotlib', 'statsmodels', 'tensorflow', 'gradio'):
    print(f'  {name:14s}', 'OK' if importlib.util.find_spec(name) else 'MISSING')
from ssq_research.scores import available_algorithms
try:
    print('算法可用性:', available_algorithms())
except Exception as probe_error:   # TF 在部分环境下导入会抛非 ImportError 的异常
    print('算法可用性探测失败（不影响后续步骤）:', repr(probe_error))
    print('提示：ARIMA / LSTM 相关步骤将不可用，其余算法照常。')

# GPU 检查：LSTM 默认参数满足 CuDNN 条件，开了 Accelerator 就会自动用，代码不用改。
if importlib.util.find_spec('tensorflow'):
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print('GPU 已启用:', [d.name for d in gpus])
    else:
        print('GPU: 无（当前是 CPU 模式）。要加速：Kaggle 右侧 Settings > Accelerator 选 GPU T4，'
              '保存后从头重跑即可，代码无需改动。')


In [ ]:
# ===== 可选：中文字体 =====
# 图表文字跟随字体可用性：检测到 CJK 字体就用中文，否则用英文，不会出现方框。
# Kaggle 默认镜像通常没有中文字体，打开下面开关会现场 apt 安装一个。
if SETUP_CJK_FONT:
    import subprocess
    installed = False
    for package in ('fonts-wqy-zenhei', 'fonts-wqy-microhei'):
        result = subprocess.run(['sudo', 'apt-get', 'install', '-y', '-qq', package],
                                capture_output=True, text=True)
        if result.returncode == 0:
            print('已安装', package)
            installed = True
            break
    if not installed:
        print('字体安装失败，图表将使用英文标签；这不影响后续任何步骤。')
    from ssq_research.viz import ensure_cjk_font, reset_font_cache
    reset_font_cache()          # 清掉导入时可能已缓存的「无字体」结论
    print('当前可用中文字体:', ensure_cjk_font() or '无')
else:
    from ssq_research.viz import has_cjk_font
    print('SETUP_CJK_FONT=False：按现有字体决定图表语言，中文字体可用 =', has_cjk_font())


In [ ]:
# ===== 取数与完整性检查 =====
from datetime import datetime, timezone
from ssq_research.data_fetch import fetch_ssq_draws_with_fallback
from ssq_research.core import load_csv

if DATA_CSV:
    draws = load_csv(pathlib.Path(DATA_CSV))
    data_source, parse_stats, attempts = DATA_CSV, None, ()
    print('已指定 DATA_CSV，跳过在线取数:', DATA_CSV)
else:
    outcome = fetch_ssq_draws_with_fallback(timeout=30)
    draws, data_source = outcome.draws, outcome.source
    parse_stats, attempts = outcome.parse_stats, outcome.attempts
    if attempts:
        print('以下数据源失败，已按顺序跳过：')
        for line in attempts:
            print('  -', line)
    if outcome.is_snapshot:
        print('警告：所有在线源都不可用，已回退到仓库内置快照。')
        print(f'      快照最新为 {outcome.latest_issue}（{outcome.latest_date}），其后的开奖不在其中。')

print('数据源:', data_source)
print('抓取时间 UTC:', datetime.now(timezone.utc).isoformat())
if not draws:
    raise RuntimeError('取数结果为空：数据源格式可能已变更，请改用 DATA_CSV 兜底。')
print('历史期数:', len(draws), '范围:', draws[0].issue, '->', draws[-1].issue)

# 完整性检查：解析统计 + 重复期号。数据脏是「发现假规律」的头号原因。
if parse_stats is not None:
    print('解析总行数:', parse_stats.total_lines, '有效期数:', parse_stats.parsed_lines,
          '跳过行数:', parse_stats.skipped_line_count)
    print('跳过原因:', parse_stats.skipped_reasons)

issues = [d.issue for d in draws]
duplicates = len(issues) - len(set(issues))
print('重复期号:', duplicates)
if duplicates:
    print('警告：存在重复期号，频次统计会被重复计数，先清数据再解读排名。')
if len(draws) < 100:
    print('警告：历史期数偏少，卡方检验的可信度有限。')


In [ ]:
# ===== 概率与显著性分析（全量样本）=====
from ssq_research.stats import build_probability_report

report = build_probability_report(draws)
print(f'样本 {report.samples} 期：{report.first_issue} -> {report.last_issue}')
print(f'红球 卡方 chi2={report.red_uniformity.statistic:.2f} '
      f'df={report.red_uniformity.degrees_of_freedom} p={report.red_uniformity.p_value:.4f}')
print(f'蓝球 卡方 chi2={report.blue_uniformity.statistic:.2f} '
      f'df={report.blue_uniformity.degrees_of_freedom} p={report.blue_uniformity.p_value:.4f}')
print()

print('红球频次 Top5（理论出现率 6/33 = 18.18%；[偏离] 表示超出单球 95% 区间）：')
for rank, row in enumerate(report.red_top, start=1):
    low, high = row.interval
    flag = ' [偏离]' if row.deviates_significantly else ''
    print(f'  {rank}. 红球 {row.number:02d}  {row.count:4d} 次  {row.rate*100:6.2f}%  '
          f'(偏差 {row.deviation*100:+.2f}%, 95% 区间 {low*100:.2f}%~{high*100:.2f}%){flag}')
print()

print('蓝球频次 Top5（理论出现率 1/16 = 6.25%）：')
for rank, row in enumerate(report.blue_top, start=1):
    low, high = row.interval
    flag = ' [偏离]' if row.deviates_significantly else ''
    print(f'  {rank}. 蓝球 {row.number:02d}  {row.count:4d} 次  {row.rate*100:6.2f}%  '
          f'(偏差 {row.deviation*100:+.2f}%, 95% 区间 {low*100:.2f}%~{high*100:.2f}%){flag}')
print()

print('各奖级理论概率（由组合数算出，非硬编码）：')
for odds in report.prizes:
    print(f'  {odds.level:6s} 1 / {odds.one_in:,.0f}')
print()

if report.has_detectable_pattern:
    print('结论：至少一类球偏离均匀假设。先排查数据完整性（漏期 / 错列 / 重复期号），再解读排名。')
else:
    print('结论：两类球都与公平开奖一致。上面的排名只描述历史，不含对下一期的预测信息。')


In [ ]:
# ===== 多算法对照回测（同窗口、同成本、同一随机基线）=====
from ssq_research.core import (RandomStrategy, FrequencyStrategy, run_backtest,
                               summarize, blue_degeneracy_report)
from ssq_research.cli import algorithm_strategies

strategies = [RandomStrategy(), FrequencyStrategy()] + algorithm_strategies(include_arima=RUN_ARIMA)
print('参与策略:', [s.name for s in strategies])

ordered = sorted(draws, key=lambda d: (d.draw_date, d.issue))
tail = ordered[-(MAX_PERIODS + WINDOW):] if MAX_PERIODS > 0 else ordered
print(f'回测样本: 最近 {len(tail)} 期（窗口 {WINDOW}），实际回测期数约 {max(0, len(tail) - WINDOW)}')
print('提示：frequency 与 frequency-argmax/frequency-sample 是三个近似策略，属于有意的对照（旧实现 vs 统一评分契约）。')

rows = run_backtest(tail, strategies, window=WINDOW, seed=SEED)

print()
print(f'{"策略":<26}{"期数":>6}{"红均":>8}{"蓝均":>8}{"蓝去重":>8}{"蓝熵":>8}{"净收益":>10}')
for item in summarize(rows):
    net = item.get('net')
    net_text = '未知' if net is None else f'{net:.1f}'
    print(f'{item["strategy"]:<26}{item["trials"]:>6}{item["avg_red_hits"]:>8.3f}'
          f'{item["avg_blue_hits"]:>8.4f}{item["distinct_blue"]:>8}'
          f'{item["blue_entropy"]:>8.3f}{net_text:>10}')

print()
print('退化检测（distinct_blue 长期只有 1~3、熵远低于 4 bit 即说明输出已塌缩成常数）：')
for name in sorted({row['strategy'] for row in rows}):
    group = [row for row in rows if row['strategy'] == name]
    print(' ', name, blue_degeneracy_report(group))


In [ ]:
# ===== 可选：LSTM 顺序回测 =====
lstm_rows = []
if RUN_LSTM:
    from ssq_research.lstm import LstmConfig, walk_forward_lstm

    lstm_config = LstmConfig(window=10, hidden_units=32, layers=2, dropout=0.1,
                             epochs=LSTM_EPOCHS, batch_size=32, retrain_every=10,
                             max_predictions=LSTM_MAX_PREDICTIONS, verbose=1, seed=SEED)
    lstm_rows = walk_forward_lstm(ordered, lstm_config,
                                  min_history=max(WINDOW, lstm_config.window + 1), seed=SEED)
    print('LSTM 配置:', lstm_config)
    print('评估期数:', len(lstm_rows) // 2, '（每期 2 条记录：lstm_argmax / lstm_sample）')
    for item in summarize(lstm_rows):
        print(' ', item['strategy'], '蓝去重', item['distinct_blue'],
              '蓝熵', round(item['blue_entropy'], 3), '红均', round(item['avg_red_hits'], 3))
    for strategy_name in ('lstm_argmax', 'lstm_sample'):
        group = [row for row in lstm_rows if row['strategy'] == strategy_name]
        print(' ', strategy_name, blue_degeneracy_report(group))
else:
    print('RUN_LSTM=False：跳过 LSTM 回测。打开后大概多花数分钟（Kaggle CPU）。')


In [ ]:
# ===== 可选：LSTM 与基线的公平区间对照 =====
# LSTM 只覆盖最后 max_predictions 期，直接和 600 期基线比是不等样本比较。
# 这里把双方都截到同一段期号上再汇总。
COMPARISON_PERIODS = 300

if lstm_rows:
    target_issues = {d.issue for d in ordered[-COMPARISON_PERIODS:]}
    fair_rows = [r for r in rows if r['issue'] in target_issues]
    fair_rows += [r for r in lstm_rows if r['issue'] in target_issues]
    print(f'公平评估区间：最后 {COMPARISON_PERIODS} 期；每期 1 注、每注 2 元')
    print()
    print(f'{"策略":<26}{"期数":>6}{"红均":>8}{"蓝均":>8}{"蓝去重":>8}{"蓝熵":>8}{"净收益":>10}')
    for item in summarize(fair_rows):
        net = item.get('net')
        net_text = '未知' if net is None else f'{net:.1f}'
        print(f'{item["strategy"]:<26}{item["trials"]:>6}{item["avg_red_hits"]:>8.3f}'
              f'{item["avg_blue_hits"]:>8.4f}{item["distinct_blue"]:>8}'
              f'{item["blue_entropy"]:>8.3f}{net_text:>10}')
    print()
    print('读法：样本对齐后，若 LSTM 行的红均/蓝均没有显著高于 random，就不能说它更好；')
    print('      若 lstm_argmax 的 blue_entropy 远低于 4 bit，说明它已经退化成常数解。')
else:
    print('RUN_LSTM=False：无 LSTM 结果可对照。')


In [ ]:
# ===== 可选：最终模型候选（研究性，不代表中奖概率提升）=====
N_COMBINATIONS = 5

if RUN_LSTM:
    import numpy as np
    from ssq_research.lstm import (LstmConfig, train_model, predict_probabilities,
                                   split_probabilities, normalize_weights)
    from ssq_research.core import BLUE_ENTROPY_MAX

    final_config = LstmConfig(window=30, hidden_units=32, layers=2, dropout=0.1,
                              epochs=LSTM_EPOCHS, batch_size=32, verbose=1, seed=SEED)
    final_model = train_model(ordered, final_config, validation_split=0.0)

    # predict_probabilities 是红/蓝双头拼回 49 维的唯一入口，不要自己切片。
    probabilities = predict_probabilities(final_model, ordered[-final_config.window:], verbose=0)
    red_scores, blue_scores = split_probabilities(probabilities)
    red_weights = normalize_weights(red_scores)
    blue_weights = normalize_weights(blue_scores)
    rng = np.random.default_rng(SEED)

    # 蓝球按概率无放回抽 5 个互不相同的号。用 argsort 取前 5 是纯确定性派生，
    # 两次运行会逐位相同 —— 那正是 0.2.0 要修掉的退化现象。
    blue_pool = [int(n) for n in rng.choice(np.arange(1, 17), size=N_COMBINATIONS,
                                            replace=False, p=blue_weights)]
    candidates = []
    seen = set()
    for blue in blue_pool:
        reds = None
        for _attempt in range(20):
            drawn = tuple(sorted(int(n) for n in rng.choice(
                np.arange(1, 34), size=6, replace=False, p=red_weights)))
            if (drawn, blue) not in seen:
                reds = drawn
                break
        if reds is None:
            print(f'警告：蓝球 {blue:02d} 连续 20 次抽到重复组合，跳过该条，不补齐。')
            continue
        seen.add((reds, blue))
        candidates.append((reds, blue))

    entropy = float(-(blue_weights * np.log2(np.clip(blue_weights, 1e-12, None))).sum())
    sorted_blue = np.sort(blue_weights)[::-1]
    print(f'SEED={SEED}  训练数据: {ordered[0].issue} -> {ordered[-1].issue}')
    print('红球概率 Top10:', [(int(i + 1), round(float(red_scores[i]), 4))
                              for i in np.argsort(red_scores)[::-1][:10]])
    print('蓝球概率 Top5:', [(int(i + 1), round(float(blue_weights[i]), 6))
                             for i in np.argsort(blue_weights)[::-1][:5]])
    print(f'蓝球分布经验熵: {entropy:.4f} bit（上限 {BLUE_ENTROPY_MAX:.4f} bit，即均匀 1/16）')
    print(f'蓝球 Top1 与 Top5 概率差: {float(sorted_blue[0] - sorted_blue[4]):.6f}'
          '（越接近 0 说明模型输出越平，越没有可用信号）')
    print()
    print(f'{len(candidates)} 组研究性候选（蓝球为按概率无放回抽样，不是 Top-N 排序）：')
    for number, (reds, blue) in enumerate(candidates, start=1):
        print(f'  {number}. 红球 {reds} + 蓝球 {blue:02d}')
    print()
    print('声明：以上是历史样本上的模型输出，不构成任何中奖概率上的优势。')
else:
    print('RUN_LSTM=False：跳过最终模型候选。')


In [ ]:
# ===== 生成单文件 HTML 仪表盘 =====
from ssq_research.report import build_report_html
from IPython.display import FileLink, display

# 第一个参数必须传全量 draws，不能传回测用的 tail（那是按 MAX_PERIODS 截断的切片）：
# 报告内部会自己重算频次表和卡方，传切片会让 HTML 里的样本数与上面打印的对不上。
document = build_report_html(draws, rows, seed=SEED, window=WINDOW)

report_path = pathlib.Path('/kaggle/working') / 'ssq_report.html'
if not report_path.parent.exists():      # 非 Kaggle 环境落到当前目录
    report_path = pathlib.Path('ssq_report.html')
report_path.write_text(document, encoding='utf-8')
print('报告已生成:', report_path.resolve(), f'{report_path.stat().st_size / 1024:.1f} KB')
print(f'报告中频次/卡方基于全量 {len(draws)} 期；多算法对照表基于最近 {MAX_PERIODS} 期回测。')
print('图表文字：检测到中文字体就用中文，否则自动用英文（不会出现方框）。')
display(FileLink(str(report_path)))


In [ ]:
# ===== 可选：Gradio 实时界面 =====
if RUN_WEBAPP:
    from ssq_research.webapp import launch
    launch(draws, share=True)
else:
    print('RUN_WEBAPP=False：需要浏览器里实时查询时改为 True（依赖 gradio）。')
    print('Kaggle 不暴露入站端口，必须 share=True 才能访问；静态 HTML 报告是更稳的产物。')


## 结果怎么读

- **排名不等于预测。** 高频号码描述的是历史。要判断「热号」是否有意义，看它是否超出单球 95% 区间；
  即便超出，同时检验 33 个号时出现一两个这种情况本身就在期望之内（多重比较）。
- **卡方 p 值大** 说明历史分布与均匀一致。这时任何「冷热号」结论都只是噪声。
- **集成学习不提高中奖概率。** 它降低的是多个模型输出之间的分歧，不是开奖本身的随机性。
  多个都打不过随机基线的模型，加权平均后依然打不过。
- **净收益里的「未知」** 指一/二等奖金额依赖奖池，不能记 0 元计入，否则中头奖反而显示为亏损。
- **先看退化指标再看命中。** `distinct_blue` 长期很小、`blue_entropy` 远低于 4 bit 时，输出已经塌缩成常数，
  命中数只是噪声。
- **两种 LSTM 策略分开读。** `lstm_sample` 带种子抽样，蓝球随 `SEED` 变化，说明抽样真的按概率走；
  `lstm_argmax` 是确定性点估计，蓝球恒定是正常现象 —— 恒定恰恰说明概率分布接近均匀、模型没有可用信号。
- **同比才有意义。** 任何策略都必须与同票数、同成本的随机基线比较，单次命中或最高奖级不能作为优势证据。

复现方法：把日志里打印的 `SEED` 填回配置格，重跑即可得到同一组结果。
